# Notebook 01: Embedding Analysis (Level 1)

## Overview
This notebook analyzes the **input embedding space** of Pythia models to determine whether
frequency band structure is already present before any transformer processing. This is the
foundation for all subsequent representational analyses.

## Key Questions
- Are tokens from different frequency bands geometrically separated in embedding space?
- Can a linear probe predict frequency band from embeddings alone?
- How does embedding structure scale with model size?
- Are there confounds (token length, special characters) that could explain any separation?

## Hypothesis Domain: R1 (Embedding Structure)
- **H-R1.1**: k-NN purity > chance (permutation test, per model)
- **H-R1.2**: Linear probe accuracy > chance (permutation test, per model)
- **H-R1.3**: Embedding norms correlate with frequency rank (Spearman, per model)
- **H-R1.4**: Separation ratio increases with model size (Spearman across models)
- **H-R1.5**: CKA between adjacent bands > distant bands (paired test, per model)
- **H-R1.6**: Intrinsic dimensionality differs by band (Kruskal-Wallis, per model)

## Notebook Structure
1. Setup & Data Loading
2. Token Property Verification (confound controls)
3. Frequency Distribution Verification
4. Embedding Norms by Band
5. Intrinsic Dimensionality (MLE)
6. Isotropy & Participation Ratio
7. k-NN Purity + Permutation Test
8. k-NN Band Confusion Matrix
9. Linear Probing (5-fold CV) + MLP Comparison
10. CKA Similarity Matrix
11. Centroid Distances & Separation Ratio
12. Principal Angles
13. Cross-Model Comparison
14. Draw Stability

## Data Sources
- Pre-extracted activations: `outputs/extraction/activations/*.npz`
- LSC datasets: `LSC_data/datasets/matched/`
- Token pools: `LSC_data/lsc_token_pools/matched/`

## 1. Setup & Data Loading

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_BANDS,
    FREQUENCY_RANK,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    DATASETS_BASE,
    TOKEN_POOLS_BASE,
    ACTIVATIONS_DIR,
    ANALYSIS_DIR,
    VIZ_DIR,
    RANDOM_SEED,
    N_PERMUTATIONS,
    K_NEIGHBORS,
    CV_FOLDS,
    get_domain_dirs,
)
from utils.data_loading import (
    load_lsc_dataset,
    load_extracted_activations,
    load_token_pool,
    build_representational_df,
    save_analysis,
    NumpyEncoder,
)
from utils.geometry import (
    compute_band_centroids,
    compute_centroid_distances,
    compute_within_band_spread,
    compute_separation_ratio,
    compute_participation_ratio,
    compute_isotropy,
    mle_intrinsic_dim,
    linear_cka,
    compute_principal_angles,
    compute_knn_purity,
    compute_knn_band_distribution,
    permutation_test_purity,
)
from utils.probing import (
    train_probe,
    train_mlp_probe,
    train_probe_with_permutation,
)
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_metric_heatmap,
    plot_boxplot_by_band,
    plot_cka_matrix,
    plot_knn_confusion,
    plot_scaling_panel,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("embedding", "base")
from functools import partial as _partial

save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")
print(f"Analysis dir: {ANALYSIS_DIR}")
print(f"Viz dir: {VIZ_DIR}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']
Analysis dir: LSC_circuit_analysis/03_Phase_Representational/outputs/embedding/base/analysis
Viz dir: LSC_circuit_analysis/03_Phase_Representational/outputs/embedding/base/viz


In [2]:
# Load all extracted activations (token embeddings)
# We need embeddings: shape (N, 21, d_model): use first source token (pos 0) as representative
all_embeddings = {}  # model -> draw -> {band: embeddings}
all_datasets = {}  # (draw, band) -> dataset

for model in MODELS:
    all_embeddings[model] = {}
    for draw in DRAWS:
        all_embeddings[model][draw] = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                # Use embeddings of unique tokens (S1-S5 at positions 0-4)
                # For embedding analysis, use the first source token (pos 0 = S1)
                # as the representative embedding per example
                all_embeddings[model][draw][band] = data["token_embeddings"][
                    :, 0, :
                ]  # (N, d_model)

                if (draw, band) not in all_datasets:
                    all_datasets[(draw, band)] = load_lsc_dataset(draw, band)
            except FileNotFoundError as e:
                print(f"  Missing: {model}/{band}/{draw}: {e}")

print(f"\nLoaded embeddings for {len(all_embeddings)} models")
for model in MODELS:
    n = sum(len(v) for v in all_embeddings[model].values())
    print(f"  {model}: {n} band-draw configurations")


Loaded embeddings for 5 models
  pythia-70m: 15 band-draw configurations
  pythia-160m: 15 band-draw configurations
  pythia-410m: 15 band-draw configurations
  pythia-1b: 15 band-draw configurations
  pythia-1.4b: 15 band-draw configurations


## 2. Token Property Verification

Verify that confound controls are intact: all tokens should be lowercase,
with comparable character lengths across bands. This is a sanity check since
the LSC data pipeline already controlled for these.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m")

token_properties = []

for band in BANDS:
    try:
        pool = load_token_pool(band)
    except FileNotFoundError:
        # Fall back to extracting from dataset
        ds = load_lsc_dataset("draw_1", band)
        unique_ids = set()
        for ex in ds["examples"]:
            unique_ids.update(ex["token_ids"][:5])  # Source tokens S1-S5
        pool = {"token_ids": list(unique_ids)}

    token_ids = (
        pool["token_ids"]
        if isinstance(pool, dict) and "token_ids" in pool
        else pool.get("tokens", [])
    )
    if isinstance(token_ids, list) and len(token_ids) > 0:
        for tid in token_ids[:200]:  # Cap at 200 for efficiency
            if isinstance(tid, dict):
                tok_id = tid.get("token_id", tid.get("id", 0))
                tok_str = tid.get("token", tokenizer.decode([tok_id]))
                freq = tid.get("frequency", tid.get("count", 0))
            else:
                tok_id = int(tid)
                tok_str = tokenizer.decode([tok_id])
                freq = 0

            token_properties.append(
                {
                    "band": band,
                    "token_id": tok_id,
                    "token_str": tok_str,
                    "char_length": len(tok_str.strip()),
                    "is_lowercase": tok_str.strip().islower()
                    if tok_str.strip().isalpha()
                    else True,
                    "is_alpha": tok_str.strip().isalpha(),
                    "frequency": freq,
                }
            )

df_tokens = pd.DataFrame(token_properties)
print(
    f"Token properties: {len(df_tokens)} tokens across {df_tokens['band'].nunique()} bands"
)
print(f"\nCharacter length by band:")
print(df_tokens.groupby("band")["char_length"].describe().round(2))
print(f"\nLowercase fraction by band:")
print(df_tokens.groupby("band")["is_lowercase"].mean())

Token properties: 1000 tokens across 5 bands

Character length by band:
           count  mean   std  min  25%  50%  75%   max
band                                                  
control    200.0  5.93  1.86  2.0  5.0  6.0  7.0  12.0
high       200.0  6.49  2.14  3.0  5.0  6.0  8.0  12.0
low        200.0  5.93  1.86  2.0  5.0  6.0  7.0  12.0
medium     200.0  5.99  2.06  2.0  4.0  6.0  7.0  13.0
very_high  200.0  6.30  2.06  3.0  5.0  6.0  8.0  13.0

Lowercase fraction by band:
band
control      1.0
high         1.0
low          1.0
medium       1.0
very_high    1.0
Name: is_lowercase, dtype: float64


In [4]:
# Kruskal-Wallis test on character length across bands
from scipy import stats as sp_stats

groups = [
    df_tokens[df_tokens["band"] == b]["char_length"].values
    for b in BANDS
    if b in df_tokens["band"].unique()
]
if len(groups) >= 2 and all(len(g) > 0 for g in groups):
    stat, p = sp_stats.kruskal(*groups)
    print(f"Kruskal-Wallis on char_length across bands: H={stat:.2f}, p={p:.4f}")
    if p < 0.05:
        print("  WARNING: Significant difference in character length across bands!")
    else:
        print("  OK: No significant difference: confound controlled.")

save_analysis(df_tokens, "01_token_properties.csv")

Kruskal-Wallis on char_length across bands: H=11.25, p=0.0239


## 3. Frequency Distribution Verification

Verify that the frequency bands are well-separated in log-frequency space.

In [5]:
# Plot frequency distribution by band (if frequency info available)
if "frequency" in df_tokens.columns and df_tokens["frequency"].sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    for band in BANDS:
        band_data = df_tokens[
            (df_tokens["band"] == band) & (df_tokens["frequency"] > 0)
        ]
        if len(band_data) > 0:
            ax.hist(
                np.log10(band_data["frequency"]),
                bins=30,
                alpha=0.5,
                color=BAND_COLORS.get(band, "gray"),
                label=BAND_NAMES.get(band, band),
            )
    ax.set_xlabel("Log10(Frequency)")
    ax.set_ylabel("Count")
    ax.set_title("Token Frequency Distribution by Band")
    ax.legend()
    save_figure(fig, "viz_01_01_frequency_distribution.png")
else:
    print("Frequency data not available in token pools: skipping distribution plot.")
    print(
        "Band separation is defined by the LSC data pipeline (see project_context.md)."
    )

Frequency data not available in token pools: skipping distribution plot.
Band separation is defined by the LSC data pipeline (see project_context.md).


## 4. Embedding Norms by Band

Do frequency bands occupy different magnitude regions in embedding space?
L2 norms of embeddings may correlate with token frequency.

In [6]:
norm_records = []

for model in MODELS:
    for draw in DRAWS:
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is None:
                continue
            norms = np.linalg.norm(emb, axis=1)  # (N,)
            for n in norms:
                norm_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "l2_norm": float(n),
                    }
                )

df_norms = pd.DataFrame(norm_records)
print(f"Embedding norms: {len(df_norms)} entries")
print(df_norms.groupby(["model", "band"])["l2_norm"].describe().round(3))

Embedding norms: 16875 entries
                       count   mean    std    min    25%    50%    75%    max
model       band                                                             
pythia-1.4b control    675.0  0.989  0.007  0.944  0.985  0.990  0.994  1.007
            high       675.0  0.995  0.004  0.977  0.992  0.995  0.998  1.007
            low        675.0  0.982  0.007  0.926  0.978  0.982  0.986  1.002
            medium     675.0  0.993  0.004  0.979  0.990  0.993  0.996  1.002
            very_high  675.0  0.987  0.008  0.947  0.983  0.988  0.993  1.005
pythia-160m control    675.0  0.795  0.009  0.757  0.790  0.796  0.802  0.818
            high       675.0  0.802  0.007  0.778  0.798  0.802  0.806  0.818
            low        675.0  0.795  0.007  0.741  0.791  0.795  0.799  0.811
            medium     675.0  0.803  0.006  0.782  0.800  0.803  0.806  0.821
            very_high  675.0  0.793  0.009  0.757  0.788  0.793  0.799  0.811
pythia-1b   control    675.0  1.0

In [7]:
# Boxplot: norms by band, per model
for model in MODELS:
    model_data = df_norms[df_norms["model"] == model]
    if len(model_data) == 0:
        continue
    fig = plot_boxplot_by_band(
        model_data,
        y="l2_norm",
        title=f"Embedding L2 Norms: {model}",
        ylabel="L2 Norm",
    )
    save_figure(fig, f"viz_01_02_embedding_norms_{model}.png")

save_analysis(
    df_norms.groupby(["model", "band"])["l2_norm"]
    .agg(["mean", "std", "median"])
    .reset_index(),
    "01_embedding_norms.csv",
)

LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


## 5. Intrinsic Dimensionality

MLE intrinsic dimensionality (Levina & Bickel, 2005) per band per model.
Do low-frequency tokens span more or fewer dimensions?

In [8]:
dim_records = []

for model in MODELS:
    for draw in DRAWS:
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is None or len(emb) < 20:
                continue
            try:
                dim = mle_intrinsic_dim(emb)
                dim_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "intrinsic_dim": dim,
                        "d_model": MODEL_D_MODEL[model],
                    }
                )
            except Exception as e:
                print(f"  MLE failed for {model}/{draw}/{band}: {e}")

df_dim = pd.DataFrame(dim_records)
print(f"Intrinsic dimensionality: {len(df_dim)} entries")
print(df_dim.groupby(["model", "band"])["intrinsic_dim"].describe().round(1))

LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


...

LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])
LSC_circuit_analysis/03_Phase_Representational/utils/geometry.py:235: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(dk[:, None] / distances[:len(dk), :k - 1])


Intrinsic dimensionality: 75 entries
                       count   mean   std    min    25%    50%    75%     max
model       band                                                             
pythia-1.4b control      3.0  673.0  42.9  638.0  649.0  660.0  690.5   720.9
            high         3.0  698.1  37.2  663.8  678.3  692.9  715.3   737.7
            low          3.0  933.7  44.1  882.9  919.2  955.5  959.0   962.5
            medium       3.0  890.5  76.7  802.4  864.4  926.3  934.5   942.7
            very_high    3.0  572.4  74.2  524.8  529.7  534.6  596.3   658.0
pythia-160m control      3.0  493.3  24.6  469.2  480.7  492.2  505.3   518.4
            high         3.0  524.9  40.4  479.5  509.1  538.7  547.7   556.7
            low          3.0  680.6  21.4  659.9  669.5  679.1  690.9   702.7
            medium       3.0  674.0  56.4  614.5  647.7  680.9  703.8   726.6
            very_high    3.0  434.0  54.6  397.7  402.6  407.5  452.1   496.8
pythia-1b   control      3.

In [9]:
# Heatmap: intrinsic dim by model x band
if len(df_dim) > 0:
    pivot = df_dim.groupby(["model", "band"])["intrinsic_dim"].mean().reset_index()
    pivot_wide = pivot.pivot(index="model", columns="band", values="intrinsic_dim")
    pivot_wide = pivot_wide.reindex(index=MODELS, columns=BANDS)
    fig = plot_metric_heatmap(
        pivot_wide, "Intrinsic Dimensionality (MLE)", fmt=".1f", cmap="viridis"
    )
    save_figure(fig, "viz_01_03_intrinsic_dim_heatmap.png")

save_analysis(df_dim, "01_dimensionality.csv")

## 6. Isotropy & Participation Ratio

How uniformly are embeddings distributed in direction space?

In [10]:
iso_records = []

for model in MODELS:
    for draw in DRAWS:
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is None or len(emb) < 10:
                continue
            pr = compute_participation_ratio(emb)
            iso = compute_isotropy(emb)
            iso_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "band": band,
                    "participation_ratio": pr,
                    "isotropy": iso,
                }
            )

df_iso = pd.DataFrame(iso_records)
print(
    df_iso.groupby(["model", "band"])[["participation_ratio", "isotropy"]]
    .mean()
    .round(3)
)

                       participation_ratio  isotropy
model       band                                    
pythia-1.4b control                154.262     0.964
            high                   136.709     0.968
            low                    142.579     0.971
            medium                 140.596     0.972
            very_high              142.919     0.959
pythia-160m control                130.779     0.956
            high                   118.162     0.959
            low                    124.813     0.959
            medium                 123.152     0.961
            very_high              120.952     0.953
pythia-1b   control                155.888     0.965
            high                   137.947     0.970
            low                    144.361     0.972
            medium                 142.625     0.974
            very_high              143.584     0.961
pythia-410m control                137.666     0.955
            high                   124.208    

## 7. k-NN Purity + Permutation Test

Are same-band tokens clustered in embedding space?

In [11]:
knn_records = []

# kNN permutation test: 200 permutations sufficient for p < 0.005 resolution
KNN_N_PERMS = min(200, N_PERMUTATIONS)
print(f"Using {KNN_N_PERMS} permutations for kNN purity test")

for model in MODELS:
    for draw in DRAWS:
        # Combine all bands for this model x draw
        embs = []
        labels = []
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is not None:
                embs.append(emb)
                labels.extend([band] * len(emb))

        if len(embs) < 2:
            continue

        X = np.vstack(embs)
        y = np.array(labels)

        # k-NN purity
        purity = compute_knn_purity(X, y, k=K_NEIGHBORS)

        # Permutation test
        p_value, null_dist = permutation_test_purity(
            X,
            y,
            observed_purity=purity,
            k=K_NEIGHBORS,
            n_permutations=KNN_N_PERMS,
            seed=RANDOM_SEED,
        )

        knn_records.append(
            {
                "model": model,
                "draw": draw,
                "purity": purity,
                "p_value": p_value,
                "null_mean": float(np.mean(null_dist)),
                "null_std": float(np.std(null_dist)),
                "n_samples": len(X),
            }
        )
        print(
            f"  {model}/{draw}: purity={purity:.3f}, p={p_value:.4f}, null={np.mean(null_dist):.3f}"
        )

df_knn = pd.DataFrame(knn_records)
save_analysis(df_knn, "01_knn_purity.csv")
print(f"\nk-NN purity results: {len(df_knn)} entries")
print(df_knn.groupby("model")[["purity", "p_value", "null_mean"]].mean().round(4))

Using 200 permutations for kNN purity test


  pythia-70m/draw_1: purity=0.308, p=0.0000, null=0.214


  pythia-70m/draw_2: purity=0.315, p=0.0000, null=0.214


  pythia-70m/draw_3: purity=0.323, p=0.0000, null=0.216


  pythia-160m/draw_1: purity=0.304, p=0.0000, null=0.214


...

  pythia-1b/draw_2: purity=0.327, p=0.0000, null=0.215


  pythia-1b/draw_3: purity=0.336, p=0.0000, null=0.216


  pythia-1.4b/draw_1: purity=0.320, p=0.0000, null=0.214


  pythia-1.4b/draw_2: purity=0.327, p=0.0000, null=0.215


  pythia-1.4b/draw_3: purity=0.336, p=0.0000, null=0.216

k-NN purity results: 15 entries
             purity  p_value  null_mean
model                                  
pythia-1.4b  0.3279      0.0     0.2148
pythia-160m  0.3112      0.0     0.2149
pythia-1b    0.3257      0.0     0.2148
pythia-410m  0.3183      0.0     0.2146
pythia-70m   0.3153      0.0     0.2148


## 8. k-NN Band Confusion Matrix

Full P(neighbor_band | token_band) matrix reveals which bands are most/least
distinguishable in embedding space.

In [12]:
for model in MODELS:
    # Use draw_1 as representative
    embs, labels = [], []
    for band in BANDS:
        emb = all_embeddings.get(model, {}).get("draw_1", {}).get(band)
        if emb is not None:
            embs.append(emb)
            labels.extend([band] * len(emb))

    if len(embs) < 2:
        continue

    X = np.vstack(embs)
    y = np.array(labels)

    confusion = compute_knn_band_distribution(X, y, k=K_NEIGHBORS, bands=BANDS)
    fig = plot_knn_confusion(
        confusion, BANDS, title=f"k-NN Band Confusion: {model} (Embeddings)"
    )
    save_figure(fig, f"viz_01_04_knn_confusion_{model}.png")

    # Save confusion matrix
    df_conf = pd.DataFrame(confusion, index=BANDS, columns=BANDS)
    save_analysis(df_conf, f"01_knn_confusion_{model}.csv")

## 9. Linear Probing + MLP Comparison

Can a linear probe predict frequency band from embeddings?

In [13]:
probe_records = []

for model in MODELS:
    for draw in DRAWS:
        embs, labels = [], []
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is not None:
                embs.append(emb)
                labels.extend([band] * len(emb))

        if len(embs) < 2:
            continue

        X = np.vstack(embs)
        y = np.array(labels)

        # Linear probe (5-fold CV)
        linear_result = train_probe(X, y)

        # MLP probe for comparison
        mlp_result = train_mlp_probe(X, y)

        # Analytical chance level for reference
        chance = 1.0 / len(BANDS)

        probe_records.append(
            {
                "model": model,
                "draw": draw,
                "linear_accuracy": linear_result["accuracy"],
                "linear_std": linear_result["std"],
                "mlp_accuracy": mlp_result["accuracy"],
                "mlp_std": mlp_result["std"],
                "linearity_gap": mlp_result["accuracy"] - linear_result["accuracy"],
                "above_chance": linear_result["accuracy"] - chance,
            }
        )
        print(
            f"  {model}/{draw}: linear={linear_result['accuracy']:.3f} (+/- {linear_result['std']:.3f}), "
            f"MLP={mlp_result['accuracy']:.3f}, gap={mlp_result['accuracy'] - linear_result['accuracy']:.3f}"
        )

df_probe = pd.DataFrame(probe_records)
save_analysis(df_probe, "01_probe_accuracy.csv")
print(f"\nProbe results: {len(df_probe)} entries")
print(f"Chance level: {1.0 / len(BANDS):.3f}")
print(
    df_probe.groupby("model")[["linear_accuracy", "mlp_accuracy", "linearity_gap"]]
    .mean()
    .round(3)
)
print("\nNote: Formal permutation significance tests are in NB07 (inferential).")

  pythia-70m/draw_1: linear=0.488 (+/- 0.026), MLP=0.460, gap=-0.028


  pythia-70m/draw_2: linear=0.462 (+/- 0.017), MLP=0.428, gap=-0.035


  pythia-70m/draw_3: linear=0.494 (+/- 0.027), MLP=0.476, gap=-0.019


  pythia-160m/draw_1: linear=0.471 (+/- 0.037), MLP=0.437, gap=-0.034


  pythia-160m/draw_2: linear=0.488 (+/- 0.030), MLP=0.466, gap=-0.022


...

  pythia-1b/draw_2: linear=0.480 (+/- 0.016), MLP=0.475, gap=-0.005


  pythia-1b/draw_3: linear=0.527 (+/- 0.030), MLP=0.525, gap=-0.002


  pythia-1.4b/draw_1: linear=0.512 (+/- 0.014), MLP=0.506, gap=-0.006


  pythia-1.4b/draw_2: linear=0.515 (+/- 0.030), MLP=0.496, gap=-0.019


  pythia-1.4b/draw_3: linear=0.538 (+/- 0.023), MLP=0.506, gap=-0.032

Probe results: 15 entries
Chance level: 0.200
             linear_accuracy  mlp_accuracy  linearity_gap
model                                                    
pythia-1.4b            0.521         0.503         -0.019
pythia-160m            0.491         0.468         -0.023
pythia-1b              0.508         0.497         -0.012
pythia-410m            0.499         0.490         -0.009
pythia-70m             0.481         0.455         -0.027

Note: Formal permutation significance tests are in NB07 (inferential).


## 10. CKA Similarity Matrix

Representational similarity between bands via CKA with bootstrap CIs.

In [14]:
cka_records = []

for model in MODELS:
    for draw in DRAWS:
        band_embs = {}
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is not None:
                band_embs[band] = emb

        if len(band_embs) < 2:
            continue

        # Compute pairwise CKA
        bands_present = [b for b in BANDS if b in band_embs]
        n_bands = len(bands_present)
        cka_matrix = np.zeros((n_bands, n_bands))

        for i, b1 in enumerate(bands_present):
            for j, b2 in enumerate(bands_present):
                if i <= j:
                    # Ensure same number of samples for CKA
                    n = min(len(band_embs[b1]), len(band_embs[b2]))
                    cka_val = linear_cka(band_embs[b1][:n], band_embs[b2][:n])
                    cka_matrix[i, j] = cka_val
                    cka_matrix[j, i] = cka_val

        # Save per-model CKA
        if draw == "draw_1":  # Plot for draw_1
            fig = plot_cka_matrix(
                cka_matrix, bands_present, title=f"CKA Similarity: {model} (Embeddings)"
            )
            save_figure(fig, f"viz_01_05_cka_{model}.png")

        # Record pairwise values
        for i, b1 in enumerate(bands_present):
            for j, b2 in enumerate(bands_present):
                if i < j:
                    # FREQUENCY_RANK maps 'control' -> None, so use `or 0` for safe subtraction
                    rank1 = FREQUENCY_RANK.get(b1) or 0
                    rank2 = FREQUENCY_RANK.get(b2) or 0
                    cka_records.append(
                        {
                            "model": model,
                            "draw": draw,
                            "band_1": b1,
                            "band_2": b2,
                            "cka": cka_matrix[i, j],
                            "is_adjacent": abs(rank1 - rank2) == 1,
                        }
                    )

df_cka = pd.DataFrame(cka_records)
save_analysis(df_cka, "01_cka_matrices.csv")
print(f"CKA results: {len(df_cka)} pairwise comparisons")
print(f"Adjacent bands CKA: {df_cka[df_cka['is_adjacent']]['cka'].mean():.3f}")
print(f"Distant bands CKA: {df_cka[~df_cka['is_adjacent']]['cka'].mean():.3f}")

CKA results: 150 pairwise comparisons
Adjacent bands CKA: 0.575
Distant bands CKA: 0.582


## 11. Centroid Distances & Separation Ratio

L2 and cosine distances between band centroids. Scale-invariant separation
ratio = mean inter-band distance / mean within-band spread.

In [15]:
sep_records = []

for model in MODELS:
    for draw in DRAWS:
        embs, labels = [], []
        for band in BANDS:
            emb = all_embeddings.get(model, {}).get(draw, {}).get(band)
            if emb is not None:
                embs.append(emb)
                labels.extend([band] * len(emb))

        if len(embs) < 2:
            continue

        X = np.vstack(embs)
        y = np.array(labels)

        # Compute separation metrics
        centroids = compute_band_centroids(X, y)
        distances = compute_centroid_distances(centroids)
        spreads = compute_within_band_spread(X, y)
        sep_ratio = compute_separation_ratio(distances, spreads)

        sep_records.append(
            {
                "model": model,
                "draw": draw,
                "separation_ratio": sep_ratio,
                "mean_centroid_distance": float(
                    distances.values[np.triu_indices_from(distances.values, k=1)].mean()
                ),
                "mean_within_spread": float(np.mean(list(spreads.values())))
                if spreads
                else 0,
            }
        )

df_sep = pd.DataFrame(sep_records)
save_analysis(df_sep, "01_separation_ratios.csv")
print(
    df_sep.groupby("model")[
        ["separation_ratio", "mean_centroid_distance", "mean_within_spread"]
    ]
    .mean()
    .round(3)
)

             separation_ratio  mean_centroid_distance  mean_within_spread
model                                                                    
pythia-1.4b             7.515                   0.162               0.022
pythia-160m             4.603                   0.130               0.028
pythia-1b               7.421                   0.172               0.023
pythia-410m             5.566                   0.136               0.025
pythia-70m              3.891                   0.120               0.031


## 12. Principal Angles

Angular divergence between band subspaces (top-k principal components per band).

In [16]:
angle_records = []

for model in MODELS:
    band_embs = {}
    for band in BANDS:
        emb = all_embeddings.get(model, {}).get("draw_1", {}).get(band)
        if emb is not None and len(emb) >= 10:
            band_embs[band] = emb

    bands_present = [b for b in BANDS if b in band_embs]
    for i, b1 in enumerate(bands_present):
        for j, b2 in enumerate(bands_present):
            if i < j:
                angles = compute_principal_angles(
                    band_embs[b1], band_embs[b2], n_components=10
                )
                angle_records.append(
                    {
                        "model": model,
                        "band_1": b1,
                        "band_2": b2,
                        "mean_angle_deg": float(np.mean(np.degrees(angles))),
                        "max_angle_deg": float(np.max(np.degrees(angles))),
                        "min_angle_deg": float(np.min(np.degrees(angles))),
                    }
                )

df_angles = pd.DataFrame(angle_records)
save_analysis(df_angles, "01_principal_angles.csv")
if len(df_angles) > 0:
    print(df_angles.groupby("model")["mean_angle_deg"].describe().round(1))

             count    mean    std     min     25%     50%     75%     max
model                                                                    
pythia-1.4b   10.0  4458.0  195.9  4025.8  4410.4  4536.4  4579.8  4634.4
pythia-160m   10.0  4196.8  233.5  3701.8  4116.5  4294.5  4341.7  4431.5
pythia-1b     10.0  4504.0  183.5  4091.3  4452.0  4588.7  4602.5  4669.4
pythia-410m   10.0  4267.0  229.9  3733.9  4214.2  4362.8  4406.0  4457.4
pythia-70m    10.0  3915.9  243.6  3423.4  3802.2  3989.3  4060.8  4233.0


## 13. Cross-Model Comparison

All metrics as functions of model size. Do larger models produce more
separated embedding spaces?

In [17]:
# Build master embedding DataFrame
master_records = []

for model in MODELS:
    for draw in DRAWS:
        record = {"model": model, "draw": draw, "model_capacity": MODEL_CAPACITY[model]}

        # Add kNN purity
        knn_row = df_knn[(df_knn["model"] == model) & (df_knn["draw"] == draw)]
        if len(knn_row) > 0:
            record["knn_purity"] = knn_row.iloc[0]["purity"]
            record["knn_p_value"] = knn_row.iloc[0]["p_value"]

        # Add probe accuracy
        probe_row = df_probe[(df_probe["model"] == model) & (df_probe["draw"] == draw)]
        if len(probe_row) > 0:
            record["probe_accuracy"] = probe_row.iloc[0]["linear_accuracy"]
            record["mlp_accuracy"] = probe_row.iloc[0]["mlp_accuracy"]
            record["linearity_gap"] = probe_row.iloc[0]["linearity_gap"]

        # Add separation ratio
        sep_row = df_sep[(df_sep["model"] == model) & (df_sep["draw"] == draw)]
        if len(sep_row) > 0:
            record["separation_ratio"] = sep_row.iloc[0]["separation_ratio"]

        master_records.append(record)

df_master = pd.DataFrame(master_records)
save_analysis(df_master, "01_master_embedding.csv")
print(
    df_master.groupby("model")[["knn_purity", "probe_accuracy", "separation_ratio"]]
    .mean()
    .round(3)
)

             knn_purity  probe_accuracy  separation_ratio
model                                                    
pythia-1.4b       0.328           0.521             7.515
pythia-160m       0.311           0.491             4.603
pythia-1b         0.326           0.508             7.421
pythia-410m       0.318           0.499             5.566
pythia-70m        0.315           0.481             3.891


In [18]:
# Scaling panel: key metrics vs model capacity
if len(df_master) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    metrics = ["knn_purity", "probe_accuracy", "separation_ratio"]
    titles = ["k-NN Purity", "Linear Probe Accuracy", "Separation Ratio"]

    for ax, metric, title in zip(axes, metrics, titles):
        if metric not in df_master.columns:
            continue
        for model in MODELS:
            mdata = df_master[df_master["model"] == model]
            if len(mdata) == 0:
                continue
            ax.scatter(
                mdata["model_capacity"],
                mdata[metric],
                color=MODEL_COLORS.get(model, "gray"),
                label=model,
                s=60,
                zorder=5,
            )
        # Mean line
        means = df_master.groupby("model")[metric].mean()
        caps = df_master.groupby("model")["model_capacity"].first()
        ax.plot(caps, means, color="black", alpha=0.3, linestyle="--")
        ax.set_xlabel("Model Capacity (M params)")
        ax.set_xscale("log")
        ax.set_title(title)

    axes[0].legend(loc="best", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "viz_01_06_scaling_panel.png")

## 14. Draw Stability

Metric variability across 3 draws: are results stable?

In [19]:
stability_records = []

for model in MODELS:
    model_data = df_master[df_master["model"] == model]
    if len(model_data) < 2:
        continue

    for metric in ["knn_purity", "probe_accuracy", "separation_ratio"]:
        if metric not in model_data.columns:
            continue
        vals = model_data[metric].dropna()
        if len(vals) < 2:
            continue
        stability_records.append(
            {
                "model": model,
                "metric": metric,
                "mean": float(vals.mean()),
                "std": float(vals.std()),
                "cv": float(vals.std() / vals.mean()) if vals.mean() != 0 else np.nan,
                "range": float(vals.max() - vals.min()),
            }
        )

df_stability = pd.DataFrame(stability_records)
save_analysis(df_stability, "01_draw_stability.csv")
print("Draw stability (CV = coefficient of variation):")
print(df_stability.pivot(index="model", columns="metric", values="cv").round(3))

Draw stability (CV = coefficient of variation):
metric       knn_purity  probe_accuracy  separation_ratio
model                                                    
pythia-1.4b       0.024           0.027             0.008
pythia-160m       0.022           0.045             0.011
pythia-1b         0.033           0.049             0.006
pythia-410m       0.028           0.044             0.007
pythia-70m        0.024           0.035             0.006


In [20]:
print("\n" + "=" * 70)
print("NOTEBOOK 01 COMPLETE")
print("=" * 70)
print(f"\nOutput CSVs in: {ANALYSIS_DIR}")
print(f"Figures in: {VIZ_DIR}")
print(f"\nKey outputs:")
for f in sorted(ANALYSIS_DIR.glob("01_*")):
    print(f"  {f.name}")
for f in sorted(VIZ_DIR.glob("viz_01_*")):
    print(f"  {f.name}")


NOTEBOOK 01 COMPLETE

Output CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/embedding/base/analysis
Figures in: LSC_circuit_analysis/03_Phase_Representational/outputs/embedding/base/viz

Key outputs:
  01_cka_matrices.csv
  01_dimensionality.csv
  01_draw_stability.csv
  01_embedding_norms.csv
  01_knn_confusion_pythia-1.4b.csv
  01_knn_confusion_pythia-160m.csv
  01_knn_confusion_pythia-1b.csv
  01_knn_confusion_pythia-410m.csv
  01_knn_confusion_pythia-70m.csv
  01_knn_purity.csv
  01_master_embedding.csv
  01_principal_angles.csv
  01_probe_accuracy.csv
  01_separation_ratios.csv
  01_token_properties.csv
  viz_01_02_embedding_norms_pythia-1.4b.png
  viz_01_02_embedding_norms_pythia-160m.png
  viz_01_02_embedding_norms_pythia-1b.png
  viz_01_02_embedding_norms_pythia-410m.png
  viz_01_02_embedding_norms_pythia-70m.png
  viz_01_03_intrinsic_dim_heatmap.png
  viz_01_04_knn_confusion_pythia-1.4b.png
  viz_01_04_knn_confusion_pythia-160m.png
  viz_01_04_knn_confusion_p